In [4]:
from monai_model import DDPMPL
import os
import torch
import torchvision
from torch.utils.data import Dataset
from torchvision import transforms
import torchvision.io
from torch.utils.data import DataLoader
from matplotlib import pyplot as plt
from datetime import datetime
from generative.inferers import DiffusionInferer
from monai.networks.schedulers import DDIMScheduler
from torchvision.datasets import ImageFolder
import pytorch_lightning as pl
from PIL import Image

device = torch.device("cuda")

In [5]:
class CustomImageDataset(Dataset):
    def __init__(self, root_dir, cond_dir, transform=None):
        self.root_dir = root_dir
        self.cond_dir = cond_dir
        self.transform = transform
        self.classes = [item for item in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir,item))]
        self.image_paths = []
        self.labels = []
        self.cond_paths = []
        
        # Collect image file paths and corresponding labels
        for label in self.classes:
            class_dir = os.path.join(root_dir, label)
            for img_name in os.listdir(class_dir):
                self.image_paths.append(os.path.join(class_dir, img_name))
                self.labels.append(self.classes.index(label))  # Assign numeric labels

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")  # Load the image

        cond_path = os.path.join(self.cond_dir, os.listdir(self.cond_dir)[0])
        condition_image = Image.open(cond_path).convert("RGB")

        if self.transform:
            image = self.transform(image)
            condition_image = self.transform(condition_image)

        label = int(self.labels[idx])       # Get the label
        

        
        return image, label, condition_image

# Data preparation
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Normalize

])
# Specify the path to your dataset
root_dir = '/mnt/raid/home/ajarry/data/trainer'
train_dir = os.path.join(root_dir,"temp_train")
val_dir = os.path.join(root_dir,"temp_val")
cond_dir = os.path.join(root_dir,"condition")

train_dataset = CustomImageDataset(root_dir=train_dir, cond_dir=cond_dir, transform=transform)
val_dataset = CustomImageDataset(root_dir=val_dir, cond_dir=cond_dir, transform=transform)


In [6]:


# Dataloader (you can mess with batch size)
batch_size = 64
num_workers = 1
train_dataloader = DataLoader(train_dataset, batch_size=batch_size,num_workers=num_workers, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size,num_workers=num_workers, shuffle=False)

# Define hyperparameters
hparams = {
    'lr': 2e-4,
    'weight_decay': 1e-5,
    'num_train_timesteps': 1000,
    'in_channels': 3,
    'out_channels': 3,
    'use_pre_trained': False
}

model = DDPMPL(**hparams)
model.to(device)

# Set up the trainer
trainer = pl.Trainer(max_epochs=100, precision=16, enable_progress_bar=True)

# Train the model
trainer.fit(model, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

Trainer will use only 1 of 4 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=4)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]

  | Name      | Type               | Params | Mode 
---------------------------------------------------------
0 | model     | DiffusionModelUNet | 18.5 M | train
1 | scheduler | DDPMScheduler      | 0      | train
---------------------------------------------------------
18.5 M    Trainable params
0         Non-trainable params
18.5 M    Total params
74.036    Total estimated model params size (MB)


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

torch.Size([64, 6, 128, 128])


RuntimeError: The size of tensor a (6) must match the size of tensor b (3) at non-singleton dimension 1

In [ ]:
# Generate a sample
model = model.to(device)
shape = (1, 3, 128, 128)  # Adjust based on model's requirements
cond = plt.imread("/mnt/raid/home/ajarry/data/trainer/condition/003f1137-63cc-4047-a265-b4aef597f980_frame_12.png")
cond = cond.unsqueeze(0).to(device)
# List to store generated samples
samples = []

# Generate 8 samples
model.eval()
with torch.no_grad():
    for _ in range(8):
        # Generate a random sample
        noise = torch.randn(shape).to(model.device)  # Noise input
        sample = model(noise).cpu().squeeze()  # Generate image and move to CPU
        samples.append(sample)

# Plot the 8 samples in a 2x4 grid
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    # Transpose the sample to (H, W, C) and convert to numpy for plotting
    img = samples[i].numpy().transpose(1, 2, 0)
    ax.imshow(img)
    ax.axis('off')

plt.tight_layout()
plt.show()